<a href="https://colab.research.google.com/github/debnowskamakrela-commits/STARTER_AI-INVEST/blob/main/Lekcja_1_analiza_spolki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Twoja pierwsza analiza — AI Invest

**Jak używać (3 kroki):**

1. W polu **TICKER** poniżej wpisz symbol swojej spółki:
   - Spółka z **USA** → sam symbol, np. `AAPL`, `MSFT`, `NVDA`
   - Spółka z **GPW** → symbol + `.WA`, np. `PKN.WA`, `CDR.WA`, `KGH.WA`
2. Najedź na komórkę i kliknij **▶** (przycisk start po lewej stronie).
3. Poczekaj chwilę — pierwsze uruchomienie może potrwać do minuty.

> Kodu nie musisz rozumieć ani nawet oglądać — Claude napisał go za Ciebie.
> Twoje jedyne zadanie: wpisać symbol i kliknąć **▶**.


In [ ]:
#@title 📊 Analiza spolki — wpisz symbol i kliknij ▶ { display-mode: "form" }

# ⬇️ Wpisz symbol spolki (USA: AAPL, MSFT, NVDA  |  GPW: PKN.WA, CDR.WA)
TICKER = "PKO.WA" #@param {type:"string"}

# ---- Ponizszego kodu nie musisz rozumiec — Claude napisal go za Ciebie ----
import subprocess, sys, warnings
warnings.filterwarnings("ignore")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yfinance"],
               capture_output=True)

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ticker = TICKER.strip().upper()

# ---------- Formatowanie (polski przecinek, "b.d." gdy brak danych) ----------
def _ok(x):
    return x is not None and not (isinstance(x, float) and (np.isnan(x) or np.isinf(x)))
def _pl(s):
    return s.replace(".", ",")
def fmt_pct(x, dec=1):
    return _pl(f"{x*100:.{dec}f}%") if _ok(x) else "b.d."
def fmt_spct(x, dec=0):
    return _pl(f"{x*100:+.{dec}f}%") if _ok(x) else "b.d."
def fmt_num(x, dec=2):
    return _pl(f"{x:.{dec}f}") if _ok(x) else "b.d."
def fmt_money(x, cur=""):
    if not _ok(x): return "b.d."
    for div, suf in [(1e12, "bln"), (1e9, "mld"), (1e6, "mln")]:
        if abs(x) >= div:
            return _pl(f"{x/div:.2f}") + f" {suf} {cur}".rstrip()
    return f"{x:,.0f}".replace(",", " ") + (f" {cur}" if cur else "")
def arrow(ch):
    if not _ok(ch): return ""
    return "📈" if ch > 0.001 else ("📉" if ch < -0.001 else "➡️")
def marg_word(curr, prev):
    if not (_ok(curr) and _ok(prev)): return ""
    d = curr - prev
    if d > 0.002:  return "poprawa 📈"
    if d < -0.002: return "spadek 📉"
    return "bez zmian ➡️"

# ---------- Pobranie danych z Yahoo Finance ----------
tk = yf.Ticker(ticker)
try:
    hist = tk.history(period="5y", auto_adjust=True)
except Exception:
    hist = pd.DataFrame()

if hist is None or hist.empty:
    print("😕 Nie udalo sie pobrac danych dla:", ticker)
    if "." not in ticker:
        print("   👉 Jesli to spolka z GPW, dodaj na koncu .WA — np. PKN.WA, CDR.WA, KGH.WA")
    else:
        print("   👉 Sprawdz, czy symbol jest poprawny (np. AAPL dla USA, PKN.WA dla GPW).")
    print("   👉 Mozesz tez kliknac ▶ jeszcze raz — czasem dane wracaja za drugim razem.")
else:
    try:
        info = tk.info
    except Exception:
        info = {}
    name      = info.get("longName") or info.get("shortName") or ticker
    price_cur = info.get("currency", "")
    fin_cur   = info.get("financialCurrency", price_cur)

    # --- Ceny / stopy zwrotu z historii ---
    close = hist["Close"].dropna()
    last_price = close.iloc[-1] if len(close) else None
    def ret_over(days):
        if len(close) < 2: return None
        past = close[close.index <= close.index[-1] - pd.Timedelta(days=days)]
        return None if past.empty else last_price / past.iloc[-1] - 1
    ret_1y = ret_over(365)

    # --- Roczne sprawozdania (rok do roku) ---
    def get_row(df, names):
        if df is None or df.empty: return None
        for n in names:
            if n in df.index:
                s = df.loc[n].dropna()
                if not s.empty: return s.sort_index()   # rosnaco po dacie
        return None
    def ratio(num, den):
        if num is None or den is None: return None
        common = num.index.intersection(den.index)
        if len(common) == 0: return None
        return (num[common] / den[common]).sort_index()
    def last_prev(s):
        if s is None or len(s) < 1: return (None, None)
        return (s.iloc[-1], s.iloc[-2] if len(s) >= 2 else None)

    try:
        fin = tk.income_stmt
    except Exception:
        fin = None

    rev_s  = get_row(fin, ["Total Revenue", "TotalRevenue", "Total Revenues"])
    opi_s  = get_row(fin, ["Operating Income", "OperatingIncome",
                           "Total Operating Income As Reported"])
    ni_s   = get_row(fin, ["Net Income", "NetIncome",
                           "Net Income Common Stockholders", "NetIncomeCommonStockholders"])
    eps_s  = get_row(fin, ["Diluted EPS", "DilutedEPS", "Basic EPS", "BasicEPS"])
    opm_s  = ratio(opi_s, rev_s)
    ntm_s  = ratio(ni_s, rev_s)

    rev, rev_prev   = last_prev(rev_s)
    eps, eps_prev   = last_prev(eps_s)
    opm, opm_prev   = last_prev(opm_s)
    ntm, ntm_prev   = last_prev(ntm_s)
    rev_year        = rev_s.index[-1].year if rev_s is not None else None

    # Fallback z info, gdy brak rocznych sprawozdan
    if not _ok(rev): rev = info.get("totalRevenue")
    if not _ok(eps): eps = info.get("trailingEps")
    if not _ok(opm): opm = info.get("operatingMargins")
    if not _ok(ntm): ntm = info.get("profitMargins")

    # Sredirocz. wzrost przychodow (CAGR) + zakres lat
    cagr, cagr_period = None, ""
    if rev_s is not None and len(rev_s) >= 2 and rev_s.iloc[0] > 0:
        yrs = len(rev_s) - 1
        cagr = (rev_s.iloc[-1] / rev_s.iloc[0]) ** (1 / yrs) - 1
        cagr_period = f"{rev_s.index[0].year}\u2013{rev_s.index[-1].year}"

    def yoy(curr, prev):
        if not (_ok(curr) and _ok(prev)) or prev == 0: return None
        return curr / prev - 1

    pe   = info.get("trailingPE")
    mcap = info.get("marketCap")
    yl   = f"({rev_year})" if rev_year else ""

    # ============================ KARTA WYNIKU ============================
    print("=" * 58)
    print(f"  {name}  ({ticker})")
    print("=" * 58)

    print("\n  CZY SPOLKA ROSNIE?")
    ch = yoy(rev, rev_prev)
    extra = f"  (rok temu {fmt_money(rev_prev, fin_cur)} \u2192 {fmt_spct(ch)} {arrow(ch)})" if _ok(rev_prev) else ""
    print(f"    Przychody {yl}: {fmt_money(rev, fin_cur)}{extra}")
    if _ok(cagr):
        print(f"    Sredirocz. wzrost przychodow: {fmt_spct(cagr, 1)} rocznie ({cagr_period})")
    ch = yoy(eps, eps_prev)
    extra = f"  (rok temu {fmt_num(eps_prev)} \u2192 {fmt_spct(ch)} {arrow(ch)})" if _ok(eps_prev) else ""
    print(f"    Zysk na akcje (EPS) {yl}: {fmt_num(eps)} {fin_cur}{extra}")

    print("\n  CZY SPOLKA ZARABIA?")
    print(f"    Marza operacyjna {yl}: {fmt_pct(opm)}"
          + (f"  (rok temu {fmt_pct(opm_prev)} \u2192 {marg_word(opm, opm_prev)})" if _ok(opm_prev) else ""))
    if _ok(opm):
        print(f"    \u2192 z kazdych 100 {fin_cur} sprzedazy ~{fmt_num(opm*100, 0)} {fin_cur} zostaje jako zysk operacyjny")
    print(f"    Marza netto {yl}: {fmt_pct(ntm)}"
          + (f"  (rok temu {fmt_pct(ntm_prev)} \u2192 {marg_word(ntm, ntm_prev)})" if _ok(ntm_prev) else ""))

    print("\n  ILE KOSZTUJE?")
    print(f"    Cena biezaca: {fmt_num(last_price)} {price_cur}"
          + (f"   (przez rok {fmt_spct(ret_1y)} {arrow(ret_1y)})" if _ok(ret_1y) else ""))
    print(f"    Kapitalizacja: {fmt_money(mcap, price_cur)}")
    print(f"    Wskaznik C/Z (P/E): {fmt_num(pe)}")
    if _ok(pe):
        print(f"    \u2192 inwestorzy placa ~{fmt_num(pe, 0)} {price_cur} za kazdy 1 {price_cur} rocznego zysku spolki")
    print("=" * 58)

    # ---------------- FLAGI: same fakty i kierunek ----------------
    flags = []
    r_ch, e_ch = yoy(rev, rev_prev), yoy(eps, eps_prev)
    if _ok(r_ch) and _ok(e_ch) and r_ch > 0 and e_ch > 0:
        flags.append("✅ Przychody i zysk rosna rok do roku")
    elif _ok(r_ch) and r_ch < 0:
        flags.append(f"⚠️ Przychody spadly rok do roku ({fmt_spct(r_ch)})")
    if _ok(opm) and _ok(opm_prev):
        if opm - opm_prev > 0.002:  flags.append("✅ Marza operacyjna poprawila sie r/r")
        elif opm - opm_prev < -0.002: flags.append("⚠️ Marza operacyjna spadla r/r")
    if _ok(ret_1y) and ret_1y < -0.15:
        flags.append(f"⚠️ Kurs spadl o {fmt_pct(abs(ret_1y))} przez ostatni rok")
    if flags:
        print("\n  Na co zwrocic uwage:")
        for f in flags[:3]:
            print("   " + f)
    print()

    # ---------------- WYKRES 1: cena 5 lat + srednia 200-dniowa ----------------
    fig, ax = plt.subplots(figsize=(10, 4.3))
    ax.plot(close.index, close.values, linewidth=1.6, label="Cena")
    ma200 = close.rolling(200).mean()
    ax.plot(ma200.index, ma200.values, linewidth=1.4, linestyle="--", label="Srednia 200-dniowa")
    ax.set_title(f"{name} — kurs z ostatnich 5 lat", fontsize=13, fontweight="bold")
    ax.set_xlabel("Rok"); ax.set_ylabel(f"Cena ({price_cur})")
    ax.legend(loc="upper left", frameon=False)
    ax.grid(True, alpha=0.25)
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    fig.tight_layout(); plt.show()

    # ---------------- WYKRES 2: przychody rok po roku ----------------
    if rev_s is not None and len(rev_s) >= 2:
        vals = rev_s.values.astype(float)
        yrs  = [d.year for d in rev_s.index]
        m = np.nanmax(np.abs(vals))
        div, unit = (1e12, "bln") if m >= 1e12 else ((1e9, "mld") if m >= 1e9 else (1e6, "mln"))
        fig2, ax2 = plt.subplots(figsize=(10, 3.1))
        bars = ax2.bar([str(y) for y in yrs], vals / div)
        ax2.set_title("Przychody rok po roku", fontsize=12, fontweight="bold")
        ax2.set_ylabel(f"Przychody ({unit} {fin_cur})")
        for b, v in zip(bars, vals / div):
            ax2.text(b.get_x() + b.get_width() / 2, v, _pl(f"{v:.1f}"),
                     ha="center", va="bottom", fontsize=9)
        ax2.grid(True, axis="y", alpha=0.25)
        fig2.tight_layout(); plt.show()
